# Amazon Bedrock Evaluations: LLM-as-a-Judge Evaluation Jobs

The earlier modules run judges in your notebook: you write the judge prompt, call the model, and parse the verdict. Amazon Bedrock also has a managed evaluation service. You give it a dataset in Amazon S3 and a list of metrics, and it runs a judge model over every record, stores per-record scores and explanations in S3, and shows a report in the Bedrock console.

This notebook runs one **LLM-as-a-judge model evaluation job** end to end. It uses the "bring your own inference responses" option: you generate the answers yourself, so the same job can grade any application or model, including ones that do not run on Bedrock.

Then it does what Module 02 taught you to do with any judge: it checks the judge against ground truth before trusting its scores.

## What you'll learn

1. When a managed evaluation job is a better fit than a judge in your notebook
2. The prompt dataset format, including your own model responses
3. How to combine built-in metrics with a custom binary (pass/fail) metric
4. The IAM service role and S3 bucket a job needs, scoped to least privilege
5. How to read the per-record results and validate the managed judge against a programmatic check

## Time and cost

The job itself takes about 10 to 20 minutes; the rest of the notebook takes a few minutes. With 15 records, four metrics and the default evaluator model, the job costs well under $1. The notebook creates one S3 bucket and one IAM role, both named with the `genai-evals-workshop` prefix, and the last section deletes them.

## Prerequisites

- Complete [Foundational Evaluations](../../Foundational%20Evaluations/), especially 02 Quality Metrics (LLM-as-judge and judge validation)
- AWS credentials allowed to create an S3 bucket and an IAM role, pass the role to Bedrock, and create Bedrock evaluation jobs (see the module README)
- Model access in `us-east-1` (or your `AWS_REGION`) to the generator model (`NOVA_2_LITE_MODEL_ID` in `Foundational Evaluations/model_config.py`) and the evaluator model (`EVALUATOR_MODEL_ID` in `evaluation_config.py`)

## 1. When to use a managed evaluation job

| | Judge in your notebook | Bedrock evaluation job |
|---|---|---|
| **Setup** | Only model access | S3 bucket and an IAM service role |
| **Judge prompt** | You write and tune it | Built-in metrics with tested prompts, plus up to 10 custom metrics |
| **Scale** | Your loop, your throttling and retries | Up to 1,000 records per job, run by the service |
| **Results** | Whatever you store | Per-record scores and explanations in S3, a report in the console |
| **Iteration speed** | Seconds | Minutes per job |
| **Judge models** | Any model | Only the [supported evaluator models](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-judge.html#evaluation-judge-supported) |

A common pattern is to design and calibrate a metric with a local judge, where iteration is fast, and then run it as a job for regular, larger evaluations whose results other people need to see.

## 2. Setup

In [ ]:
# %pip install -r requirements.txt

In [ ]:
import json
import os
import re
import sys
import time

import boto3
import pandas as pd

# The generator model comes from the workshop's shared config.
sys.path.append(os.path.abspath(os.path.join("..", "..", "Foundational Evaluations")))
from model_config import NOVA_2_LITE_MODEL_ID

# The evaluator model is set in this module, because evaluation jobs accept only supported evaluators.
from evaluation_config import EVALUATOR_MODEL_ID, RESOURCE_PREFIX
from eval_job_utils import cleanup, create_resources, read_job_results, scores_by_metric, upload_jsonl, wait_for_job

REGION = os.environ.get("AWS_REGION", "us-east-1")
GENERATOR_MODEL_ID = NOVA_2_LITE_MODEL_ID

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)

print(f"Region:          {REGION}")
print(f"Generator model: {GENERATOR_MODEL_ID}")
print(f"Evaluator model: {EVALUATOR_MODEL_ID}")

In [ ]:
# Check model access before creating anything.
for model_id in (GENERATOR_MODEL_ID, EVALUATOR_MODEL_ID):
    reply = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": "Reply with the single word OK."}]}],
        inferenceConfig={"maxTokens": 50},
    )
    text = next(b["text"] for b in reply["output"]["message"]["content"] if "text" in b)
    print(f"OK  {model_id}: {text.strip()}")

## 3. Build a dataset with ground truth

The questions come from the same US cities data as Module 02, so every question has a known answer. Ten ask for a city's population and five for its population density. Because the answers are numbers, a few lines of Python can check them, and that programmatic check is what you will later compare the judge with.

In [ ]:
cities = pd.read_csv(
    os.path.join("..", "..", "Foundational Evaluations", "02-quality-metrics", "city_pop.csv"),
    encoding="utf-8-sig", thousands=",",
)
cities["city"] = cities["city"].str.replace(r"\[.*?\]", "", regex=True).str.strip()  # drop footnote marks such as [c]
cities["density"] = cities["population"] / cities["land_area_mi2"]

# A fixed sample, spread across the list, so the run is repeatable.
sample = cities.iloc[[0, 3, 7, 12, 20, 33, 47, 68, 95, 140, 5, 18, 40, 77, 120]].reset_index(drop=True)

dataset = []
for i, row in sample.iterrows():
    place = f"{row.city}, {row.state}"
    if i < 10:
        question = f"What is the population of {place}? Give the number, then one sentence of context."
        expected = int(row.population)
        reference = f"The population of {place} is {expected:,}."
        kind = "population"
    else:
        question = (f"What is the population density of {place}, in people per square mile of land area? "
                    "Give the number, then one sentence of context.")
        expected = round(float(row.density))
        reference = f"The population density of {place} is about {expected:,} people per square mile of land area."
        kind = "density"
    dataset.append({"id": i, "kind": kind, "question": question, "expected": expected, "reference": reference})

pd.DataFrame(dataset)[["kind", "question", "expected"]].head(12)

## 4. Generate the responses you want graded

Here the generator is a small Bedrock model with no access to the data, so some of its numbers will be wrong. In your own work this step is your application: a RAG pipeline, an agent, or a model on another platform. The job never calls the generator; it only reads the responses you give it.

In [ ]:
def generate(question: str) -> str:
    reply = bedrock_runtime.converse(
        modelId=GENERATOR_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": question}]}],
        inferenceConfig={"maxTokens": 300},
    )
    return next(b["text"] for b in reply["output"]["message"]["content"] if "text" in b).strip()

for item in dataset:
    item["response"] = generate(item["question"])

print(dataset[0]["question"])
print(dataset[0]["response"])

In [ ]:
# A comma counts only as a thousands separator, so "2023," is read as 2023.
NUMBER = re.compile(r"(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)\s*(million|thousand)?", re.IGNORECASE)

def stated_number(text: str):
    """The first number in the text that is not a year, reading '8.5 million' as 8,500,000.

    Answers often start with "As of 2023, ...", so a bare four-digit number from
    1900 to 2100 is treated as a year and skipped.
    """
    for match in NUMBER.finditer(text):
        raw, unit = match.group(1), (match.group(2) or "").lower()
        value = float(raw.replace(",", "")) * {"million": 1_000_000, "thousand": 1_000}.get(unit, 1)
        if not unit and "," not in raw and "." not in raw and 1900 <= value <= 2100:
            continue
        return value
    return None

TOLERANCE = 0.05  # within 5% of the true value counts as correct
for item in dataset:
    value = stated_number(item["response"])
    item["stated"] = value
    item["programmatic_pass"] = value is not None and abs(value - item["expected"]) <= TOLERANCE * item["expected"]

checked = pd.DataFrame(dataset)
print(f"Programmatic check: {checked.programmatic_pass.sum()} of {len(checked)} answers within {TOLERANCE:.0%}")
checked[["kind", "expected", "stated", "programmatic_pass"]]

## 5. Write the prompt dataset

Each line of the JSON Lines file has:

- `prompt`: the input your application received
- `referenceResponse` (optional): the ground truth. The built-in Correctness and Completeness metrics use it when it is present.
- `category` (optional): the job reports scores per category as well as overall
- `modelResponses`: exactly one response, with a `modelIdentifier`. Every record in a job must use the same identifier, and it must match `inferenceSourceIdentifier` in the job request.

A job accepts up to 1,000 records.

In [ ]:
RESPONSE_SOURCE = "nova-2-lite-city-qa"  # names the system that produced the responses

records = [
    {
        "prompt": item["question"],
        "referenceResponse": item["reference"],
        "category": item["kind"],
        "modelResponses": [{"response": item["response"], "modelIdentifier": RESPONSE_SOURCE}],
    }
    for item in dataset
]
print(json.dumps(records[0], indent=2))

## 6. Create the S3 bucket and the service role

The job reads the dataset and writes results as a service role that Bedrock assumes. `create_resources` in `eval_job_utils.py` creates:

- **An S3 bucket** named `genai-evals-workshop-<account>-<region>`, with all public access blocked
- **An IAM role** that only Bedrock evaluation jobs in this account and Region can assume (the trust policy checks `aws:SourceAccount` and `aws:SourceArn`), with an inline policy that allows reading and writing that one bucket and invoking Bedrock models

Read the two policy functions in `eval_job_utils.py` before you run the cell. Running it again reuses the bucket and the role.

In [ ]:
resources = create_resources(REGION)
run_id = time.strftime("%Y%m%d-%H%M%S")
dataset_uri = upload_jsonl(resources, f"datasets/city-qa-{run_id}.jsonl", records)
output_uri = f"s3://{resources.bucket}/results/"
print(f"Dataset: {dataset_uri}")
print(f"Results: {output_uri}")

## 7. Choose the metrics

Three built-in metrics. Each one uses a judge prompt that Bedrock maintains for each supported evaluator model; you can read them in [Built-in metric evaluator prompts](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-type-judge-prompt.html). Bedrock normalizes the scores to the range 0 to 1.

| Metric | What it asks |
|---|---|
| `Builtin.Correctness` | Is the answer correct? Uses `referenceResponse` when present |
| `Builtin.Completeness` | Does the answer address every part of the question? |
| `Builtin.FollowingInstructions` | Did the answer follow the format the prompt asked for (a number, then one sentence)? |

One custom metric, written the way this workshop writes judges: one failure mode, a binary verdict, and the definition of pass in the prompt. In custom metric instructions, `{{prompt}}` is the prompt, `{{prediction}}` is the model response and `{{ground_truth}}` is the reference response. The variables must come last. The rating scale is given separately, and Bedrock adds the output-format instructions for the evaluator model.

The custom metric asks for the same 5% tolerance as the programmatic check, so you can compare the two directly.

In [ ]:
BUILTIN_METRICS = ["Builtin.Correctness", "Builtin.Completeness", "Builtin.FollowingInstructions"]
CUSTOM_METRIC = "number_within_5_percent"

custom_metric = {
    "customMetricDefinition": {
        "name": CUSTOM_METRIC,
        "instructions": (
            "You are checking a numeric answer about a US city against a reference answer.\n\n"
            "Find the main number the response states as its answer (a population or a population density). "
            "Compare it with the number in the reference answer.\n\n"
            "- Pass: the response states a number, and it is within 5 percent of the reference number. "
            "Wording, rounding such as '2.7 million', and extra context do not matter.\n"
            "- Fail: the response states no number, or the number differs from the reference number by more than 5 percent.\n\n"
            "Question: {{prompt}}\n"
            "Reference answer: {{ground_truth}}\n"
            "Response: {{prediction}}"
        ),
        "ratingScale": [
            {"definition": "Pass", "value": {"floatValue": 1.0}},
            {"definition": "Fail", "value": {"floatValue": 0.0}},
        ],
    }
}

## 8. Create the evaluation job

`inferenceConfig` uses `precomputedInferenceSource` instead of a Bedrock model, which tells the job to grade the responses in the dataset rather than call a model.

In [ ]:
evaluator = {"bedrockEvaluatorModels": [{"modelIdentifier": EVALUATOR_MODEL_ID}]}
job_name = f"{RESOURCE_PREFIX}-city-qa-{run_id}"

job_arn = bedrock.create_evaluation_job(
    jobName=job_name,
    jobDescription="GenAI evaluations workshop: city Q&A graded with built-in and custom metrics",
    roleArn=resources.role_arn,
    applicationType="ModelEvaluation",
    evaluationConfig={
        "automated": {
            "datasetMetricConfigs": [{
                "taskType": "General",
                "dataset": {"name": "city-qa", "datasetLocation": {"s3Uri": dataset_uri}},
                "metricNames": BUILTIN_METRICS + [CUSTOM_METRIC],
            }],
            "evaluatorModelConfig": evaluator,
            "customMetricConfig": {"customMetrics": [custom_metric], "evaluatorModelConfig": evaluator},
        }
    },
    inferenceConfig={"models": [{"precomputedInferenceSource": {"inferenceSourceIdentifier": RESPONSE_SOURCE}}]},
    outputDataConfig={"s3Uri": output_uri},
)["jobArn"]

print(f"Created {job_name}")
print(f"Console: https://{REGION}.console.aws.amazon.com/bedrock/home?region={REGION}#/eval/evaluation")

⏱️ The next cell waits for the job, which usually takes 10 to 20 minutes. If your kernel restarts, set `job_arn` to the ARN printed above and run the cell again.

In [ ]:
job = wait_for_job(bedrock, job_arn)
print(job["status"], job.get("failureMessages", ""))

## 9. Read the results

The job writes one output record per input record, under `results/<job name>/<job id>/`. Each output record holds the input record and, under `automatedEvaluationResult.scores`, one entry per metric with the score and the judge's explanation.

In [ ]:
results = read_job_results(resources, output_uri, job_arn)
print(f"{len(results)} output records")
print(json.dumps(results[0], indent=2)[:2500])

In [ ]:
METRICS = BUILTIN_METRICS + [CUSTOM_METRIC]
by_prompt = {item["question"]: item for item in dataset}

rows = []
for record in results:
    prompt = record["inputRecord"]["prompt"]
    scores = scores_by_metric(record)
    row = {"id": by_prompt[prompt]["id"], "kind": by_prompt[prompt]["kind"],
           "programmatic_pass": by_prompt[prompt]["programmatic_pass"]}
    for metric in METRICS:
        row[metric] = (scores.get(metric) or {}).get("result")
        row[f"{metric} explanation"] = (scores.get(metric) or {}).get("explanation", "")
    rows.append(row)

scored = pd.DataFrame(rows).sort_values("id").reset_index(drop=True)
summary = scored.groupby("kind")[METRICS].mean().T
summary["overall"] = scored[METRICS].mean()
summary.round(2)

## 10. Validate the managed judge

A managed judge is still a judge. Module 02 measured a judge's true positive rate (TPR: of the answers that are correct, how many the judge passes) and true negative rate (TNR: of the answers that are wrong, how many the judge fails) against human labels. Here the labels come from the programmatic check, which knows the true numbers.

Correctness is on a scale, so a score of 1.0 counts as a pass; the custom metric is already binary.

In [ ]:
def agreement(judge_pass: pd.Series, truth: pd.Series) -> dict:
    tp = int((judge_pass & truth).sum()); tn = int((~judge_pass & ~truth).sum())
    fp = int((judge_pass & ~truth).sum()); fn = int((~judge_pass & truth).sum())
    return {
        "agree": f"{tp + tn}/{len(truth)}",
        "TPR": round(tp / (tp + fn), 2) if tp + fn else None,
        "TNR": round(tn / (tn + fp), 2) if tn + fp else None,
        "false passes": fp,
        "false fails": fn,
    }

truth = scored.programmatic_pass.astype(bool)
pd.DataFrame({
    CUSTOM_METRIC: agreement(scored[CUSTOM_METRIC] >= 0.5, truth),
    "Builtin.Correctness (score = 1.0)": agreement(scored["Builtin.Correctness"] >= 0.999, truth),
}).T

In [ ]:
# Read every disagreement: is the judge wrong, or is the programmatic check?
for _, row in scored.iterrows():
    item = dataset[row.id]
    judge_pass = row[CUSTOM_METRIC] >= 0.5
    if judge_pass != row.programmatic_pass:
        print(f"[{row.id}] expected {item['expected']:,}, stated {item['stated']}, "
              f"programmatic={'pass' if row.programmatic_pass else 'fail'}, judge={'pass' if judge_pass else 'fail'}")
        print(f"  Response: {item['response'][:300]}")
        print(f"  Judge:    {row[CUSTOM_METRIC + ' explanation'][:500]}\n")
if all((scored[CUSTOM_METRIC] >= 0.5) == scored.programmatic_pass):
    print("The custom metric agrees with the programmatic check on every record.")

What to look for:

- **False passes** matter most: the judge accepts a wrong number. A high TNR is what makes a judge safe to gate releases on.
- A disagreement is not always the judge's fault. A regular expression can pick up the wrong number in a response (a year, or a second figure); read the response before you decide which side is wrong.
- The built-in Correctness metric was not told about the 5% tolerance, so it may call a close-but-different number partly correct. That is a reason to write a custom metric when your definition of correct is specific.

## 11. Try it yourself

1. Change `GENERATOR_MODEL_ID` to `DEFAULT_MODEL_ID` from the shared config, rerun from section 4, and compare the scores. Run a second job rather than editing the first one: jobs are immutable records you can compare in the console.
2. Remove `referenceResponse` from the records and rerun. How does Builtin.Correctness change when the judge has no ground truth?
3. Add a second custom metric for a different failure mode, for example "the context sentence states a fact that is false".

## 12. Clean up

This deletes the evaluation jobs whose names start with `genai-evals-workshop`, the S3 bucket and everything in it (the dataset and the results), and the IAM role. Download anything you want to keep first.

In [ ]:
cleanup(REGION, bedrock)

## Conclusion

You ran a managed LLM-as-a-judge evaluation over responses you generated yourself, combined built-in metrics with a custom binary metric, and checked the judge against ground truth before reading its scores as results.

### Key takeaways

- **Bring your own responses** and one job format grades any application, on Bedrock or elsewhere.
- **Built-in metrics** are a fast start; a **custom metric** encodes your definition of correct.
- **Validate the judge**, managed or not: measure TPR and TNR against labels you trust, and read the disagreements.

### Learn more

- [Evaluate model performance using another LLM as a judge](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-judge.html)
- [Create a prompt for a custom metric](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-custom-metrics-prompt-formats.html)
- [Evaluate a knowledge base (RAG evaluation jobs)](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-kb.html), the same service for retrieval and RAG